<a href="https://colab.research.google.com/github/aldo02032004/NER_env/blob/main/notebooks/01_label.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Label NER: GLiNER + validasi Gemini -> data latih

1. **Runtime -> Change runtime type -> T4 GPU -> Save**
2. Jalankan Cell ①. Kalau diminta restart, restart lalu lanjut dari Cell ②.
3. Isi form di Cell ②, klik Cell ②, lalu **Runtime -> Run cell and below**.

Kuota Gemini habis di Cell ⑥? Tunggu, lalu jalankan ulang Cell ⑥ saja: hasil sebelumnya tidak diulang.

In [ ]:
# @title ① Install (sekali per session)
import importlib.util, os, subprocess, sys

REPO_DIR = "/content/NER_env"
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    print("[INFO] Repo sudah ada, di-update (git pull)")
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/aldo02032004/NER_env.git", REPO_DIR], check=True)
    print(f"[INFO] Repo di-clone ke {REPO_DIR}")

# paket yang sudah ter-import lalu di-update = wajib restart
already_loaded = [p for p in ["gliner", "transformers", "great"] if p in sys.modules]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

missing = [m for m in ["torch", "gliner", "great", "emoji", "ftfy", "google.genai"] if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f"[ERROR] Paket gagal terpasang: {missing}. Scroll ke atas, cari error pip-nya.")
commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"[INFO] Python {sys.version.split()[0]} | NER_env commit {commit}")
if already_loaded:
    print(f"[PERLU RESTART] {already_loaded} sudah ter-import sebelumnya. Runtime -> Restart session, lalu lanjut dari Cell ②.")
else:
    print("[OK] Semua paket siap. Isi form di Cell ②, lalu Runtime -> Run cell and below.")

In [ ]:
# @title ② Form & config
# @markdown **PROJECT**: nama singkat (huruf kecil, tanpa spasi). Dipakai untuk nama file dan field `project` di data latih.
PROJECT = "prabowo_vladivostok"  # @param {type:"string"}
# @markdown **LINKS**: link Google Sheets/Drive, pisahkan dengan koma. Akses file harus *Anyone with the link*.
LINKS = ""  # @param {type:"string"}
# @markdown **CONFIG_PATH**: config bersama semua proyek (label, threshold, alias).
CONFIG_PATH = "/content/drive/MyDrive/ner/config.json"  # @param {type:"string"}

import re, sys
from google.colab import drive
drive.mount("/content/drive")
REPO_DIR = "/content/NER_env"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from ner.config import load_config

PROJECT = re.sub(r"[^a-z0-9_]+", "_", PROJECT.strip().lower()).strip("_")
problems = [msg for ok, msg in [(PROJECT, "PROJECT masih kosong"), (LINKS.strip(), "LINKS masih kosong")] if not ok]
if problems:
    raise ValueError("[ERROR] Form belum lengkap: " + "; ".join(problems))

cfg = load_config(CONFIG_PATH)
print(f"[OK] PROJECT = {PROJECT}")
print(f"[INFO] Label ({len(cfg['labels'])}): {cfg['labels']}")
print(f"[INFO] Model: {cfg['active_model']} | threshold review {cfg['ner_review_threshold'][cfg['active_model']]} "
      f"| sampel {cfg['review_sample_rate']} | sampel tanpa entitas {cfg['no_entity_sample_rate']}")
print(f"[INFO] LLM: {cfg['llm_provider']} / {cfg['llm_model']} | batas per run: {cfg['llm_budget_per_run']}")
print(f"[INFO] Prompt GLiNER: {cfg['gliner_label_prompts']}")
print(f"[INFO] Alias proyek ini: {len(cfg['aliases'].get(PROJECT, {}))} entri")

In [ ]:
# @title ③ Load spreadsheet & cleaning (tidak ada baris yang dihapus)
from IPython.display import display
from ner.io import load_sheet, prepare_frame

df_sheet = load_sheet(LINKS)
print(f"[INFO] Data dimuat: {df_sheet.shape[0]} baris, kolom: {list(df_sheet.columns)}")
df = prepare_frame(df_sheet)

print(f"[INFO] Akun media     : {df['is_media'].sum()}")
print(f"[INFO] Retweet        : {df['is_retweet'].sum()}")
print(f"[INFO] Terlalu pendek : {df['too_short'].sum()}")
print(f"[INFO] Duplikat       : {df['is_duplicate'].sum()}")
ready = df[df["siap_ner"]]
docs = dict(zip(ready["doc_id"], ready["text"]))
print(f"[OK] {len(docs)} dari {len(df)} baris siap NER")
display(df[["text_raw", "text", "is_media", "is_retweet", "too_short", "is_duplicate", "siap_ner"]].head(10))

In [ ]:
# @title ④ GLiNER (hasil disimpan, bisa resume)
import torch
from gliner import GLiNER
from ner.formats import spans_to_names
from ner.predict import predict_all, preds_cache_path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("[WARN] GLiNER jalan di CPU, bisa BERJAM-JAM. Ganti ke T4 GPU lalu mulai lagi dari Cell ①.")
gliner_model = GLiNER.from_pretrained(cfg["active_model"]).to(DEVICE)

cache = preds_cache_path(cfg, PROJECT)
preds = predict_all(gliner_model, docs, cfg, cache)
n_ent = sum(len(s) for s in preds.values())
print(f"[OK] {n_ent} entitas di {sum(bool(s) for s in preds.values())} doc, {sum(not s for s in preds.values())} doc tanpa entitas")
print(f"[INFO] Cache: {cache}")
for d in list(docs)[:5]:
    ents = {k: v for k, v in spans_to_names(docs[d], preds[d], cfg["labels"]).items() if v}
    print(f"\n- {docs[d][:150]}\n  {ents}")

In [ ]:
# @title ⑤ Routing: doc mana yang dicek Gemini
from ner.routing import doc_score, route_all

routes = route_all(docs, preds, cfg)
for reason in ("low_score", "no_entity_capital", "no_entity_sample", "random_sample"):
    sample = [d for d, r in routes.items() if r == reason][:2]
    for d in sample:
        print(f"[{reason}] skor={doc_score(preds[d])} | {docs[d][:120]}")

In [ ]:
# @title ⑥ Gemini -> data latih (bisa resume saat kuota habis)
from google.colab import userdata
from ner.llm import log_path, make_generate, print_summary, review_all, summarize_log
from ner.schema import read_jsonl

api_key = None
if cfg["llm_provider"] == "gemini_api":
    try:
        api_key = userdata.get("GOOGLE_API_KEY")
    except Exception as e:
        raise RuntimeError(
            f"[ERROR] Secret GOOGLE_API_KEY belum bisa dibaca ({type(e).__name__}). Cara memperbaiki:\n"
            "  1. Klik ikon kunci (Secrets) di sidebar kiri Colab\n"
            "  2. Add new secret. Name: GOOGLE_API_KEY, Value: API key dari https://aistudio.google.com/apikey\n"
            "  3. Nyalakan toggle 'Notebook access' di baris GOOGLE_API_KEY, lalu jalankan ulang cell ini\n"
            "  (atau pakai Colab AI: ubah llm_provider di config menjadi \"colab\")") from None
generate = make_generate(cfg, api_key)
LOG = log_path(cfg, PROJECT)
MASTER = cfg["paths"]["master_data"]

finished = review_all(docs, preds, routes, cfg, PROJECT, generate, MASTER, LOG)
log_rows = read_jsonl(LOG)
print("\n=== Ringkasan per route_reason ===")
print_summary(summarize_log(log_rows, routes))
master_rows = [r for r in read_jsonl(MASTER) if r["project"] == PROJECT]
print(f"\n[INFO] Data latih proyek ini di master: {len(master_rows)} doc ({MASTER})")
missing = [r for r in log_rows if r["status"] == "missing" and r["doc_id"] in routes][:5]
for r in missing:
    print(f"[missing] {r['missing']} | {docs[r['doc_id']][:120]}")
print("[OK] Selesai" if finished else "[STOP] Belum selesai, jalankan ulang cell ini setelah kuota pulih")

In [ ]:
# @title ⑦ Export per proyek (kompatibel pipeline lama)
import os
from ner.io import build_export
from ner.llm import llm_entities

THRESHOLD = cfg["ner_review_threshold"][cfg["active_model"]]
out = build_export(df, docs, preds, llm_entities(read_jsonl(LOG)), cfg["labels"], cfg["aliases"].get(PROJECT, {}),
                   min_score=THRESHOLD)   # entitas GLiNER yang belum dicek Gemini: hanya yang skornya >= threshold
out_dir = cfg["paths"]["output_dir"]
os.makedirs(out_dir, exist_ok=True)
out.to_pickle(f"{out_dir}/{PROJECT}_entities.pkl")     # kolom list tetap list
out.to_csv(f"{out_dir}/{PROJECT}_entities.csv", index=False)
print(f"[OK] {len(out)} baris ditulis ke {out_dir}/{PROJECT}_entities.pkl dan .csv")
print(f"[INFO] ner_source: {out['ner_source'].value_counts(dropna=False).to_dict()}")
for lab in cfg["labels"]:
    names = out[f"entities_{lab}"].explode().dropna()
    print(f"  {lab:<20} {len(names):>6} sebutan | teratas: {names.value_counts().head(5).to_dict()}")
display(out.head(10))